### Add complete ORF1B p-distance values to the table with PCR-fragment p-distance values

In [30]:
import pandas as pd

pcr = pd.read_csv("pairs_nt_pdist_gt02_pcr_genbank_headers.csv")
comp = pd.read_csv("pairs_complete_orf1b_genbank_headers.csv")

pcr = pcr.rename(columns={"value.x": "nt p-distance pcr", "value.y": "aa p-distance pcr"})
comp = comp.rename(columns={"value.x": "nt p-distance complete", "value.y": "aa p-distance complete"})

merged = pcr.merge(comp, on=["row", "col"], how="left")

merged.to_csv("pairs_nt_pdist_gt02_pcr_and_complete_genbank_headers.csv", index=False)


print(merged.head())
print("rows:", len(merged), "| missing complete:", merged["nt p-distance complete"].isna().sum())

                                                 row  \
0                 LC047793/JPN/NA/Bos-taurus/2009/NA   
1  MZ819168/CHL/PoAstV5-Swine-CHI-FB016-2017/Sus-...   
2      OP963600/CHN/NA/Hipposideros-larvatus/2015/NA   
3  PQ678017/CHN/LJNL-YNSC78/Crocidura-fuliginosa/...   
4  MZ819170/CHL/PoAstV5-Swine-CHI-FB036-2017/Sus-...   

                                                 col  nt p-distance pcr  \
0  PQ678017/CHN/LJNL-YNSC78/Crocidura-fuliginosa/...           0.535014   
1      OP963600/CHN/NA/Hipposideros-larvatus/2015/NA           0.535014   
2  MZ819168/CHL/PoAstV5-Swine-CHI-FB016-2017/Sus-...           0.535014   
3                 LC047793/JPN/NA/Bos-taurus/2009/NA           0.535014   
4      OP963600/CHN/NA/Hipposideros-larvatus/2015/NA           0.532213   

   aa p-distance pcr  nt p-distance complete  aa p-distance complete  
0           0.512605                0.506667                0.545263  
1           0.504202                0.483062                0.526423  

### Check species relations (same or different)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("pairs_nt_pdist_gt02_pcr_and_complete_genbank_headers.csv", sep=",")

def get_species(name):
    if pd.isna(name):
        return None
    parts = str(name).split("/")
    sp = parts[-1].strip()
    #print(sp)
    if sp == "" or sp == "NA" or sp.lower() == "nan":
        return None
    return sp

df["species_row"] = df["row"].apply(get_species)
df["species_col"] = df["col"].apply(get_species)

def species_relation(r, c):
    if r is None or c is None:
        return "unknown"
    return "same" if r == c else "different"

df["species_relation"] = [
    species_relation(r, c) for r, c in zip(df["species_row"], df["species_col"])
]

df[["row", "species_row", "col", "species_col", "species_relation"]]

,row,species_row,col,species_col,species_relation
0,LC047793/JPN/NA/Bos-taurus/2009/NA,NaN,PQ678017/CHN/LJNL-YNSC78/Crocidura-fuliginosa/...,NaN,different
1,MZ819168/CHL/PoAstV5-Swine-CHI-FB016-2017/Sus-...,NaN,OP963600/CHN/NA/Hipposideros-larvatus/2015/NA,NaN,different
2,OP963600/CHN/NA/Hipposideros-larvatus/2015/NA,NaN,MZ819168/CHL/PoAstV5-Swine-CHI-FB016-2017/Sus-...,NaN,different
3,PQ678017/CHN/LJNL-YNSC78/Crocidura-fuliginosa/...,NaN,LC047793/JPN/NA/Bos-taurus/2009/NA,NaN,different
4,MZ819170/CHL/PoAstV5-Swine-CHI-FB036-2017/Sus-...,NaN,OP963600/CHN/NA/Hipposideros-larvatus/2015/NA,NaN,different
...,...,...,...,...,...
1039587,PX137011/CHN/PAstV1-NMBB11-2023/Sus-scrofa/202...,NaN,KF499111/HKG/NA/Felis-catus/2012/Mamastrovirus...,Mamastrovirus-felis,different
1039588,PX137011/CHN/PAstV1-NMBB11-2023/Sus-scrofa/202...,NaN,MK671310/CHN/MAV2-18CC0502/Felis-catus/2018/Ma...,Mamastrovirus-felis,different
1039589,PX137011/CHN/PAstV1-NMBB11-2023/Sus-scrofa/202...,NaN,NC_022249/HKG/NA/Felis-catus/2012/Mamastroviru...,Mamastrovirus-felis,different
1039590,PX137011/CHN/PAstV1-NMBB11-2023/Sus-scrofa/202...,NaN,ON595824/AUS/FPV-1/Felis-catus/2017/Mamastrovi...,Mamastrovirus-felis,different


In [32]:
df["species_relation"].value_counts()

species_relation
different    1035320
same            4272
Name: count, dtype: int64

### Only keep rows with same species

In [ ]:
mask_same = df["species_relation"] == "same"         
mask_dist = df["nt p-distance complete"] > 0.2

same_species_gt02 = df[mask_same & mask_dist].copy()

print(f"same species: {mask_same.sum()}")
print(f"из них nt complete > 0.2: {len(same_species_gt02)}")
same_species_gt02

same species: 4272
из них nt complete > 0.2: 4056


,row,col,nt p-distance pcr,aa p-distance pcr,nt p-distance complete,aa p-distance complete,species_row,species_col,species_relation
6397,KP747574/CHN/AstV-LL-2/Sus-scrofa/2006/Mamastr...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,0.491525,0.542373,0.477571,0.532091,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
6514,MK378498/CHN/PoAstV_VIRES_GZ04_C6/Sus-scrofa/2...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,0.491525,0.508475,0.481405,0.530992,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
6550,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,KP747574/CHN/AstV-LL-2/Sus-scrofa/2006/Mamastr...,0.491525,0.542373,0.477571,0.532091,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
6601,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,MK378498/CHN/PoAstV_VIRES_GZ04_C6/Sus-scrofa/2...,0.491525,0.508475,0.481405,0.530992,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
8593,KP747574/CHN/AstV-LL-2/Sus-scrofa/2006/Mamastr...,OL695850/CHN/AH4-3/Sus-scrofa/2021/Mamastrovir...,0.488701,0.533898,0.470790,0.523711,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
...,...,...,...,...,...,...,...,...,...
1039127,OL695854/CHN/AH2-4/Sus-scrofa/2021/Mamastrovir...,MW504551/USA/15211_AST_len_5924_cov_1028.51303...,0.205333,0.128000,0.205434,0.121272,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
1039354,KP747573/CHN/AstV-LL-1/Sus-scrofa/2006/Mamastr...,MW504551/USA/15211_AST_len_5924_cov_1028.51303...,0.202667,0.112000,0.212061,0.125249,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
1039391,MW504551/USA/15211_AST_len_5924_cov_1028.51303...,KP747573/CHN/AstV-LL-1/Sus-scrofa/2006/Mamastr...,0.202667,0.112000,0.212061,0.125249,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same
1039393,MW504551/USA/15211_AST_len_5924_cov_1028.51303...,MW504561/USA/15215_AST_len_6433_cov_73.331765/...,0.202667,0.128000,0.203446,0.133201,Mamastrovirus-suisorientalis,Mamastrovirus-suisorientalis,same


In [34]:
same_species_gt02["species_row"].value_counts()

species_row
Mamastrovirus-suisorientalis     1970
Mamastrovirus-suisvulgaris       1168
Mamastrovirus-bovisorientalis     418
Mamastrovirus-felis               412
Mamastrovirus-ovis                 86
Mamastrovirus-zalophi               2
Name: count, dtype: int64

In [ ]:
cols_to_save = ["row", "col", "species_row",
                "nt p-distance pcr", "aa p-distance pcr",
                "nt p-distance complete", "aa p-distance complete"]
same_species_gt02[cols_to_save].to_csv(
    "same_species_nt_complete_gt02_genbank_headers.csv", index=False
)

Сохранено → same_species_nt_complete_gt02_genbank_headers.csv


### ICTV and GenBank headers comparison

In [38]:
import pandas as pd

f1 = pd.read_csv("pairs_complete_orf1b_genbank_headers.csv") 
f2 = pd.read_csv("pairs_complete_orf1b_ictv_headers.csv")

def add_pair_key(df):
    df = df.copy()
    df["pair_key"] = [
        tuple(sorted((str(r), str(c)))) for r, c in zip(df["row"], df["col"])
    ]
    return df

f1 = add_pair_key(f1)
f2 = add_pair_key(f2)

print("file1:", len(f1), "строк,", f1["pair_key"].nunique(), "уникальных пар")
print("file2:", len(f2), "строк,", f2["pair_key"].nunique(), "уникальных пар")

file1: 1106704 строк, 553878 уникальных пар
file2: 1106704 строк, 553878 уникальных пар


In [39]:
keys1 = set(f1["pair_key"])
keys2 = set(f2["pair_key"])

only_in_f1 = keys1 - keys2
only_in_f2 = keys2 - keys1

print(f"Только в file1: {len(only_in_f1)}")
print(f"Только в file2: {len(only_in_f2)}")

Только в file1: 18783
Только в file2: 18783


In [40]:
only1_df = f1[f1["pair_key"].isin(only_in_f1)].drop_duplicates("pair_key")
only2_df = f2[f2["pair_key"].isin(only_in_f2)].drop_duplicates("pair_key")

only1_df.to_csv("only_in_file1.csv", index=False)
only2_df.to_csv("only_in_file2.csv", index=False)

only1_df

,row,col,value.x,value.y,pair_key
302,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,OR652457/CHN/stool/Capra-hircus/2023/NA,0.496266,0.541752,(GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-s...
494,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,0.494182,0.546201,(GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-s...
509,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,PX137058/CHN/PAstV5-LNBP9-2022/Sus-scrofa/2022/NA,0.494134,0.542443,(LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/N...
572,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,OP094602/CHN/B80F05/Rhizomys-pruinosus/2019/NA,0.493603,0.577778,(GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-s...
646,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,ON624252/CHN/NA/Bos-taurus/2021/NA,0.493084,0.541494,(GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-s...
...,...,...,...,...,...
1105928,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,0.000000,0.000000,(MW504573/USA/15213_AST_len_6253_cov_80.443692...
1105929,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,0.000000,0.000000,(MW504576/USA/15210_AST_len_6624_cov_2753.3250...
1105930,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,0.000000,0.000000,(MW504577/USA/15217_AST_len_6652_cov_21.318023...
1106084,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,0.000000,0.000000,(OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovi...


In [48]:
import pandas as pd

f1 = pd.read_csv("pairs_complete_orf1b_genbank_headers.csv") 
f2 = pd.read_csv("pairs_complete_orf1b_ictv_headers.csv")

assert len(f1) == len(f2), f"разная длина: {len(f1)} vs {len(f2)}"

diffs = []
for i in range(len(f1)):
    r1, c1 = f1.at[i, "row"], f1.at[i, "col"]
    r2, c2 = f2.at[i, "row"], f2.at[i, "col"]
    if r1 != r2 or c1 != c2:
        diffs.append({
            "line": i,
            "file1": f"{r1} | {c1}",
            "file2": f"{r2} | {c2}",
            "row_file1": r1, "row_file2": r2,
            "col_file1": c1, "col_file2": c2,
        })

diff_df = pd.DataFrame(diffs)
diff_df.to_csv("header_differences.csv", index=False)
print(f"Различий: {len(diff_df)}")
diff_df

Различий: 37548


,line,file1,file2,row_file1,row_file2,col_file1,col_file2
0,302,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,OR652457/CHN/stool/Capra-hircus/2023/NA,OR652457/CHN/stool/Capra-hircus/2023/NA
1,303,OR652457/CHN/stool/Capra-hircus/2023/NA | GQ50...,OR652457/CHN/stool/Capra-hircus/2023/NA | GQ50...,OR652457/CHN/stool/Capra-hircus/2023/NA,OR652457/CHN/stool/Capra-hircus/2023/NA,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
2,494,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...
3,503,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,PV400867/CHN/NA/Bos-taurus/2025/Mamastrovirus-...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
4,509,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,PX137058/CHN/PAstV5-LNBP9-2022/Sus-scrofa/2022/NA,PX137058/CHN/PAstV5-LNBP9-2022/Sus-scrofa/2022/NA
...,...,...,...,...,...,...,...
37543,1105928,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...,MW504573/USA/15213_AST_len_6253_cov_80.443692/...
37544,1105929,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...,MW504576/USA/15210_AST_len_6624_cov_2753.32501...
37545,1105930,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...
37546,1106084,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...


In [50]:
import pandas as pd

f1 = pd.read_csv("pairs_complete_orf1b_genbank_headers.csv") 
f2 = pd.read_csv("pairs_complete_orf1b_ictv_headers.csv")
assert len(f1) == len(f2)

r1, c1 = f1["row"].values, f1["col"].values
r2, c2 = f2["row"].values, f2["col"].values

dr = r1 != r2
dc = c1 != c2
mask = dr | dc

rows = []
for i in mask.nonzero()[0]:
    if dr[i]:
        rows.append((r1[i], r2[i]))
    if dc[i]:
        rows.append((c1[i], c2[i]))

# убрать обратные дубли (a,b) <-> (b,a), сохранив направление file1 -> file2
seen = set()
uniq = []
for a, b in rows:
    if (b, a) in seen:
        continue
    seen.add((a, b))
    uniq.append((a, b))

out = pd.DataFrame(uniq, columns=["file1", "file2"])
out.to_csv("header_differences.csv", index=False)
print(len(out))
out

37872


,file1,file2
0,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
1,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
2,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
3,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...,GQ502193/USA/VA2-human-Stl-WD0680-2009/Homo-sa...
4,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...,LT898424/NA/PAstV-GER-L00919-K17_14-02-2014/NA...
...,...,...
37867,MW504577/USA/15217_AST_len_6652_cov_21.318023/...,MW504577/USA/15217_AST_len_6652_cov_21.318023/...
37868,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...
37869,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...,OL695848/CHN/AH4-1/Sus-scrofa/2021/Mamastrovir...
37870,OL695850/CHN/AH4-3/Sus-scrofa/2021/Mamastrovir...,OL695850/CHN/AH4-3/Sus-scrofa/2021/Mamastrovir...
